In [ ]:
# Loading in the dataset
import h5py

# Verify Tensorflow installation and version
import tensorflow as tf
print(tf.version.VERSION)

# Loading in the datasets
import glob
import numpy as np

# For the 75/25 split
from sklearn.model_selection import train_test_split

# Confusion Matrix plotting
import matplotlib.pyplot as plt
import seaborn as sns

### Loading in the data

Data is stored as a single self-describing HDF5 file per dataset version. No per-file CSV scan, no hardcoded channel counts.

**Schema** (`dataset/synthetic_v1.h5` — written by `data_generator.ipynb`):
- `/X`: shape `(N, 4500, 30)`, float32. First dim = attempt, second = timestep, third = channel (5 IMUs × 6 axes: accelX, accelY, accelZ, gyroX, gyroY, gyroZ).
- `/y`: shape `(N,)`, int64. Class labels 0=easy, 1=medium, 2=hard.
- Root attrs: `fs` (Hz), `n_channels`, `class_names`, `generator_seed`, `generator_version`, `created_at`, `accel_channels`, `gyro_channels`.

When real climbing data comes online in W5, the writer in `data_generator.ipynb` is replaced with an ingestion script that writes `dataset/real_v1.h5` in the same schema. **This notebook's loader cell below does not change** — only the value of `DATASET_PATH`. That's the whole point of fixing on HDF5 now.

In [ ]:
# Load synthetic (or real) data from HDF5.
# Schema: /X (N, 4500, 30) float32, /y (N,) int64, root attrs include fs, class_names, etc.
# Same loader works for real data — just point DATASET_PATH at real_v1.h5 when it exists.

DATASET_PATH = 'dataset/synthetic_v1.h5'

with h5py.File(DATASET_PATH, 'r') as f:
    x_recordings = f['X'][:]                                  # (N, 4500, 30) float32
    y_recordings = f['y'][:]                                  # (N,) int64
    labels = [s.decode() for s in f.attrs['class_names']]    # ['easy', 'medium', 'hard']
    fs = int(f.attrs['fs'])
    print(f"Loaded {DATASET_PATH}")
    print(f"  version: {f.attrs['generator_version']}, created: {f.attrs['created_at']}")
    print(f"  fs: {fs} Hz, n_channels: {int(f.attrs['n_channels'])}")

print(f"\nx_recordings: {x_recordings.shape}, dtype={x_recordings.dtype}")
print(f"y_recordings: {y_recordings.shape}, class counts={np.bincount(y_recordings)}")
print(f"labels: {labels}")

### View the data???

### Splitting it up into windows???

### Pre-Processing???

### 75/25 Split

In [ ]:
# NOTE: for the synthetic pipeline check, we feed raw (4500, 30) tensors directly —
# no windowing, no per-channel normalization. The preprocessing/windowing cells above
# are stubs to fill in once we decide on the real-data strategy (W4 on the plan).
#
# For synthetic data specifically, normalization would DESTROY the variance signal
# that distinguishes classes — don't add it here without thinking through implications.

# Shuffle and split 75/25, stratified by class to guarantee all 3 classes in both splits
x_train, x_test, y_train, y_test = train_test_split(
    x_recordings, y_recordings,
    test_size=0.25,
    stratify=y_recordings,
    random_state=42)

print(f"Training samples: {x_train.shape}, class counts: {np.bincount(y_train)}")
print(f"Testing samples:  {x_test.shape}, class counts: {np.bincount(y_test)}")

### Creating and training the model

In [ ]:
## Conv1D based model
model = tf.keras.models.Sequential([
  tf.keras.Input(shape=(4500, 30)),
  tf.keras.layers.Conv1D(filters=16, kernel_size=7, padding="same", use_bias=False),
  tf.keras.layers.BatchNormalization(), # is often paired with use_bias=False
  tf.keras.layers.ReLU(),
  tf.keras.layers.MaxPool1D(pool_size=4),

  tf.keras.layers.Conv1D(filters=32, kernel_size=5, padding="same", use_bias=False),
  tf.keras.layers.BatchNormalization(),
  tf.keras.layers.ReLU(),
  tf.keras.layers.MaxPool1D(pool_size=4),
  
  tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding="same", use_bias=False),
  tf.keras.layers.BatchNormalization(),
  tf.keras.layers.ReLU(),
  tf.keras.layers.MaxPool1D(pool_size=4),

  tf.keras.layers.Conv1D(filters=64, kernel_size=3, padding="same", use_bias=False),
  tf.keras.layers.BatchNormalization(),
  tf.keras.layers.ReLU(),
  tf.keras.layers.GlobalAveragePooling1D(),

  tf.keras.layers.Dropout(0.4),
  tf.keras.layers.Dense(32, activation='relu'),
  tf.keras.layers.Dropout(0.3),
  tf.keras.layers.Dense(3, activation='softmax')
])
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy', 'precision', "F1Score"])

model.fit(x_train, y_train, epochs=30)
test_loss, test_acc = model.evaluate(x_test,  y_test, verbose=2)

print("Test loss:", test_loss)
print("Test acc:", test_acc)
model.summary()

### Confusion Matrix
Now we will run predictions using the reserved test data to see how well the training worked. Generate a heat map showing right/wrong guesses vs. truth by motion class.

In [ ]:
# Evaluate the training parameters on test data
%matplotlib inline

Y_pred = model.predict(x_test)
y_pred = np.argmax(Y_pred, axis=1)
confusion_matrix = tf.math.confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(confusion_matrix,
            annot=True,
            xticklabels=labels,
            yticklabels=labels,
            cmap=plt.cm.Blues,
            fmt='d', cbar=False)
plt.tight_layout()
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()

### Saving the model
Save it to both a 'standard' file format (.h5) and to a 'lightweight' format for use with the microcontroller (.tflite)

In [ ]:
# Save the model and move it to the microcontroller
model.save('model.keras')

# Convert to tflite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('model.tflite', 'wb') as f:
          f.write(tflite_model)